# Strong-lens cutouts: HST · JWST · Euclid + Legacy DR11 · PS1

Previous name `lens_cutouts_v2.ipynb`.

Given (RA, Dec), fetch every public **HST**, **JWST** and **Euclid** image covering the position, plus ground-based colour context from **Legacy Surveys DR11** and **Pan-STARRS1**.

**Orientation.** Every panel is shown **north up, east left**, and carries an N/E compass:
* Space images are resampled with their own WCS onto a north-up TAN grid centred on the target (native pixel scale), so drizzled/rotated HST and JWST mosaics line up with Euclid.
* Legacy JPEGs are rendered by the Legacy server on a north-up grid centred on the target.
* PS1 JPEGs come straight from the PS1 stack projection cells, which are north-up at the cell centre; away from it north can be tilted by up to a few degrees at high Dec. Fine for inspection, not for position angles.

**Sources**
* HST / JWST — MAST level-3 DRC/DRZ/I2D read from the public AWS S3 copy with byte-range requests; deepest public observation per (instrument, filter).
* Euclid — IRSA MER mosaics (VIS, Y, J, H); Q1 by default (`EUCLID_COLLECTION` in 3a).
* Legacy DR11 — `layer=ls-dr11`: *griz* colour mapping in the DECam footprint (Dec < 32.375°), *grz* in the northern DR9 area.
* PS1 — *gri* stack colour image (Dec > −30°).

**Sections:** 1 setup · 2 parameters · 3 functions · 4 one target · 5 interactive viewer · 6 save · 7 batch

## 1. Setup

In [ ]:
# Uncomment on first use (on the Rubin Science Platform add --user)
# %pip install -q astropy astroquery reproject s3fs boto3 requests pillow ipywidgets

In [ ]:
import os
import re
import warnings
from dataclasses import dataclass
from io import BytesIO

import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.nddata import Cutout2D
from astropy.stats import sigma_clipped_stats
from astropy.wcs import WCS, FITSFixedWarning
from astropy.wcs.utils import proj_plane_pixel_scales

warnings.simplefilter("ignore", FITSFixedWarning)

EUCLID_COLLECTION = "euclid_DpdMerBksMosaic"  # Q1 MER mosaics; change when DR1 is on IRSA
EUCLID_WAVE_NM = {"VIS": 715.0, "Y": 1080.0, "J": 1370.0, "H": 1770.0}
MARGIN = 1.5  # fetch this factor x size so north-up rotation leaves no blank corners

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## 2. Parameters

To switch target: edit `RA`, `DEC` (and `NAME`), re-run this cell, then sections 4–6. Or use `quicklook()` in 4b.

In [ ]:
RA, DEC   = 296.36317084, -14.88085752        # deg, or sexagesimal strings: "10:00:27.9", "+02:12:21"
NAME      = None                      # None -> J<ra><dec>
SIZE      = 10.0                      # box side [arcsec]
TELESCOPES   = ["hst", "jwst", "euclid"]
MAX_PER_BAND = 1                      # observations kept per instrument+filter (deepest first)
EUCLID_EXTERNAL = False               # also show ground-based bands packaged in Euclid MER tiles
LEGACY    = True                      # Legacy Surveys DR11 colour panel
PS1       = True                      # Pan-STARRS1 gri colour panel
OUTDIR    = "cutouts"
WORKERS   = 8                         # parallel cutout reads per target

## 3. Functions

Run once; nothing is fetched until section 4.

### 3a. Constants and data container

In [ ]:
@dataclass
class Cut:
    telescope: str
    instrument: str
    band: str
    wave_nm: float          # only used to order bands blue -> red
    data: np.ndarray        # native pixels (MARGIN x requested size)
    wcs: WCS
    source: str             # S3 URI / URL
    obs_id: str = ""
    exptime: float = np.nan

    @property
    def label(self):
        return f"{self.telescope} {self.instrument} {self.band}"

    @property
    def pixscale(self):
        return float(np.mean(proj_plane_pixel_scales(self.wcs.celestial)) * 3600.0)


def _log(msg, verbose=True):
    if verbose:
        print(msg, flush=True)


def _f(v, default=np.nan):
    """float() that tolerates masked / missing table values."""
    try:
        v = np.ma.filled(v, default)
        return float(v)
    except (TypeError, ValueError):
        return default


def _safe(s):
    return re.sub(r"[^A-Za-z0-9._+-]+", "-", str(s)).strip("-")


def parse_coord(ra, dec):
    """Accept decimal degrees or sexagesimal strings ('10:00:27.9', '+02:12:21')."""
    if isinstance(ra, str) and re.search(r"[:hms ]", ra.strip()):
        return SkyCoord(ra, dec, unit=(u.hourangle, u.deg))
    return SkyCoord(float(ra), float(dec), unit="deg")

### 3b. Remote byte-range reader

Headers through a 1 MB buffer, then only the cutout rows by byte-range request. `_has_data` rejects chip gaps / mosaic edges so the next-deepest observation is tried.

In [ ]:
def _science_hdu(hdul):
    try:
        return hdul["SCI"]
    except KeyError:
        pass
    for h in hdul:
        if h.header.get("NAXIS", 0) == 2 and WCS(h.header).has_celestial:
            return h
    raise ValueError("no 2-D celestial image HDU")


HEADER_BLOCK = 2**20      # bytes read per header request (s3fs default is 50 MB!)
ROW_SPAN_MAX = 16 * 2**20  # read whole rows in one request if their span is below this
_BITPIX = {8: "u1", 16: "i2", 32: "i4", 64: "i8", -32: "f4", -64: "f8"}
STATS = {"requests": 0, "bytes": 0}  # running transfer counter (approximate)


def _count(nbytes, nreq=1):
    STATS["requests"] += nreq
    STATS["bytes"] += nbytes


def remote_cutout(url, coord, size_arcsec):
    """Read only the pixel box around `coord` from a remote (s3/https) or local FITS image.

    Headers are parsed through a small (1 MB) read-ahead buffer; the pixel rows are then
    fetched with one byte-range request (or one small request per row, issued concurrently,
    when the image is very wide). Falls back to HDU.section for compressed/odd HDUs.
    """
    import fsspec

    kw = {"anon": True} if url.startswith("s3://") else {}
    fs, path = fsspec.core.url_to_fs(url, **kw)
    with fs.open(path, "rb", block_size=HEADER_BLOCK, cache_type="readahead") as fh:
        with fits.open(fh, lazy_load_hdus=True, memmap=False) as hdul:
            hdu = _science_hdu(hdul)
            hdr = hdu.header
            wcs = WCS(hdr, naxis=2)
            ny, nx = hdu.shape[-2:]
            pix = float(np.mean(proj_plane_pixel_scales(wcs.celestial)) * 3600.0)
            x, y = wcs.world_to_pixel(coord)
            if not (np.isfinite(x) and np.isfinite(y)):
                raise ValueError("position not on image")
            half = int(np.ceil(size_arcsec / pix / 2.0)) + 2
            xc, yc = int(round(float(x))), int(round(float(y)))
            x0, x1 = max(0, xc - half), min(nx, xc + half + 1)
            y0, y1 = max(0, yc - half), min(ny, yc + half + 1)
            if x0 >= x1 or y0 >= y1:
                raise ValueError("position not on image")
            plain = (type(hdu) in (fits.PrimaryHDU, fits.ImageHDU)
                     and hdr.get("NAXIS") == 2 and hdr.get("BITPIX") in _BITPIX)
            if not plain:  # compressed etc.: let astropy handle it
                _count(0)
                data = np.asarray(hdu.section[y0:y1, x0:x1], dtype=float)
                return data, wcs.slice((slice(y0, y1), slice(x0, x1)))
            off = hdu.fileinfo()["datLoc"]
        _count(fh.cache.total_requested_bytes if hasattr(fh.cache, "total_requested_bytes")
               else HEADER_BLOCK)

    dt = np.dtype(_BITPIX[hdr["BITPIX"]]).newbyteorder(">")
    rowb = nx * dt.itemsize
    span = (y1 - y0) * rowb
    if span <= ROW_SPAN_MAX:  # one request for the full rows, then slice columns
        buf = fs.cat_file(path, start=off + y0 * rowb, end=off + y1 * rowb)
        _count(len(buf))
        data = np.frombuffer(buf, dt).reshape(y1 - y0, nx)[:, x0:x1]
    else:  # very wide image: one small range per row (fsspec issues these concurrently)
        starts = [off + j * rowb + x0 * dt.itemsize for j in range(y0, y1)]
        ends = [s_ + (x1 - x0) * dt.itemsize for s_ in starts]
        bufs = fs.cat_ranges([path] * len(starts), starts, ends)
        _count(sum(len(b) for b in bufs), len(bufs))
        data = np.frombuffer(b"".join(bufs), dt).reshape(y1 - y0, x1 - x0)
    data = data.astype(float)
    if hdr["BITPIX"] > 0:  # integer images: BLANK, BSCALE, BZERO
        if "BLANK" in hdr:
            data[data == hdr["BLANK"]] = np.nan
        data = data * hdr.get("BSCALE", 1.0) + hdr.get("BZERO", 0.0)
    elif hdr.get("BSCALE", 1.0) != 1.0 or hdr.get("BZERO", 0.0) != 0.0:
        data = data * hdr.get("BSCALE", 1.0) + hdr.get("BZERO", 0.0)
    return data, wcs.slice((slice(y0, y1), slice(x0, x1)))


def _has_data(data, wcs, coord, box=2, min_frac=0.3):
    """Target lands on valid pixels (not a chip gap / mosaic edge)."""
    x, y = wcs.world_to_pixel(coord)
    xi, yi = int(round(float(x))), int(round(float(y)))
    sub = data[max(0, yi - box):yi + box + 1, max(0, xi - box):xi + box + 1]
    if sub.size == 0 or not np.any(np.isfinite(sub) & (sub != 0)):
        return False
    return np.mean(np.isfinite(data) & (data != 0)) >= min_frac

### 3c. MAST (HST, JWST)

One observation query and one product-list query per target; per-filter cutouts read in parallel.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

_CLOUD = {"enabled": None}
_CLOUD_LOCK = threading.Lock()


def _mast_cloud(verbose=True):
    """Enable MAST's AWS dataset once per session (not once per target)."""
    from astroquery.mast import Observations
    with _CLOUD_LOCK:
        if _CLOUD["enabled"] is None:
            try:
                Observations.enable_cloud_dataset(provider="AWS", verbose=False)
                _CLOUD["enabled"] = True
            except Exception as e:  # boto3 missing etc.
                _CLOUD["enabled"] = False
                _log(f"[MAST] S3 access unavailable ({e})", verbose)
    return _CLOUD["enabled"]


def fetch_mast(coord, size, collections=("HST", "JWST"), max_per_band=1, max_tries=4,
               workers=8, https_fallback=False, verbose=True):
    """Deepest public level-3 image per (telescope, instrument, filter).

    https_fallback: if a product is not on S3, try MAST's HTTPS download URL. Off by default
    because if that server ignores byte ranges the whole (GB-size) file is transferred.
    """
    from astroquery.mast import Observations

    cloud = _mast_cloud(verbose)
    obs = Observations.query_criteria(coordinates=coord, radius=1 * u.arcsec,
                                      obs_collection=list(collections),
                                      dataproduct_type="image", calib_level=3,
                                      intentType="science")
    if "dataRights" in obs.colnames:
        obs = obs[obs["dataRights"] == "PUBLIC"]
    _log(f"[MAST] {len(obs)} public level-3 image observations cover the position", verbose)
    if len(obs) == 0:
        return []

    groups = {}
    for row in obs:
        filt = str(row["filters"])
        if "detection" in filt.lower():
            continue
        key = (str(row["obs_collection"]), str(row["instrument_name"]), filt)
        groups.setdefault(key, []).append(row)
    for rows in groups.values():
        rows.sort(key=lambda r: -_f(r["t_exptime"], 0.0))
        del rows[max_tries:]

    # ONE product-list request for every candidate observation of this target
    cand = [str(r["obsid"]) for rows in groups.values() for r in rows]
    prods = Observations.get_product_list(cand)
    prods = Observations.filter_products(prods, productType="SCIENCE", calib_level=[3],
                                         productSubGroupDescription=["DRC", "DRZ", "I2D"])
    if "dataRights" in prods.colnames:
        prods = prods[prods["dataRights"] == "PUBLIC"]
    parent = "parent_obsid" if "parent_obsid" in prods.colnames else "obsID"
    sub_rank = {"DRC": 0, "I2D": 0, "DRZ": 1}
    by_obs = {}
    for p in prods:
        if "_total_" in str(p["productFilename"]):
            continue
        by_obs.setdefault(str(p[parent]), []).append(p)
    for oid, ps in by_obs.items():  # the observation's own product first, DRC/I2D over DRZ
        ps.sort(key=lambda p: (str(p["obsID"]) != oid,
                               sub_rank.get(str(p["productSubGroupDescription"]), 9)))

    def _uri(p):
        uri = None
        if cloud:
            try:
                with _CLOUD_LOCK:
                    uri = Observations.get_cloud_uri(p)
            except Exception:
                uri = None
        if uri is None and https_fallback:
            uri = "https://mast.stsci.edu/api/v0.1/Download/file?uri=" + str(p["dataURI"])
        return uri

    def _one_group(item):
        (tel, inst, filt), rows = item
        out, msgs = [], []
        for row in rows:
            if len(out) >= max_per_band:
                break
            ps = by_obs.get(str(row["obsid"]))
            if not ps:
                continue
            uri = _uri(ps[0])
            if uri is None:
                msgs.append(f"  {tel} {inst} {filt}: {row['obs_id']} not on S3, skipped")
                continue
            try:
                data, wcs = remote_cutout(uri, coord, size)
            except Exception as e:
                msgs.append(f"  {tel} {inst} {filt}: {row['obs_id']} failed ({e})")
                continue
            if not _has_data(data, wcs, coord):
                msgs.append(f"  {tel} {inst} {filt}: {row['obs_id']} no valid pixels, next")
                continue
            wave = 0.5 * (_f(row["em_min"]) + _f(row["em_max"]))
            out.append(Cut(tel, inst, filt, wave, data, wcs, uri, str(row["obs_id"]),
                           _f(row["t_exptime"])))
            msgs.append(f"  {tel} {inst} {filt}: {row['obs_id']} ({_f(row['t_exptime'], 0):.0f}s)")
        return out, msgs

    cuts = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for out, msgs in ex.map(_one_group, sorted(groups.items())):
            cuts += out
            for m in msgs:
                _log(m, verbose)
    return cuts

### 3d. Euclid (IRSA)

In [ ]:
def fetch_euclid(coord, size, collection=EUCLID_COLLECTION, include_external=False,
                 workers=8, verbose=True):
    from astroquery.ipac.irsa import Irsa

    tbl = Irsa.query_sia(pos=(coord, 1 * u.arcsec), collection=collection)
    if len(tbl) == 0:
        _log("[Euclid] no MER mosaic covers the position", verbose)
        return []
    tbl = tbl[np.array([str(s) == "science" for s in tbl["dataproduct_subtype"]])]
    if not include_external and "facility_name" in tbl.colnames:
        tbl = tbl[np.array([str(s) == "Euclid" for s in tbl["facility_name"]])]
    _log(f"[Euclid] {len(tbl)} science mosaics (all bands/tiles) cover the position", verbose)

    by_band = {}
    for row in tbl:
        by_band.setdefault(str(row["energy_bandpassname"]), []).append(row)

    def _one_band(item):
        band, rows = item
        msgs = []
        for row in rows:  # overlapping tiles: take the first one with valid pixels
            url = str(row["access_url"])
            try:
                data, wcs = remote_cutout(url, coord, size)
            except Exception as e:
                msgs.append(f"  Euclid {band}: {os.path.basename(url)} failed ({e})")
                continue
            if not _has_data(data, wcs, coord):
                continue
            wave = EUCLID_WAVE_NM.get(band, np.nan)
            for lo, hi in (("energy_emin", "energy_emax"), ("em_min", "em_max")):
                if np.isnan(wave) and lo in tbl.colnames:
                    wave = 0.5 * (_f(row[lo]) + _f(row[hi])) * 1e9  # m -> nm
            inst = str(row["instrument_name"]) if "instrument_name" in tbl.colnames else ""
            oid = str(row["obs_id"]) if "obs_id" in tbl.colnames else ""
            msgs.append(f"  Euclid {band}: {os.path.basename(url)}")
            return [Cut("Euclid", inst, band, wave, data, wcs, url, oid)], msgs
        return [], msgs

    cuts = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        for out, msgs in ex.map(_one_band, sorted(by_band.items())):
            cuts += out
            for m in msgs:
                _log(m, verbose)
    return cuts

### 3e. Ground-based colour images: Legacy Surveys DR11, Pan-STARRS1

Change `LEGACY_LAYER` (e.g. `"ls-dr11-grz"` for the old *grz* mapping) or `PS1_FILTERS` (e.g. `"grz"`, `"izy"`) here.

In [ ]:
LEGACY_LAYER = "ls-dr11"  # griz colour mapping in the DECam footprint, grz in the DR9-north area
PS1_FILTERS = "gri"       # PS1 colour bands (red = reddest, blue = bluest of the set)


def _jpeg(content):
    from PIL import Image
    return np.asarray(Image.open(BytesIO(content)).convert("RGB"))


def fetch_legacy(coord, size, pixscale=0.262, layer=LEGACY_LAYER):
    """Legacy Surveys colour JPEG (north up, east left, centred on target). None if no data."""
    import requests

    n = min(512, max(16, int(np.ceil(size / pixscale))))
    url = ("https://www.legacysurvey.org/viewer/jpeg-cutout"
           f"?ra={coord.ra.deg:.6f}&dec={coord.dec.deg:.6f}&layer={layer}"
           f"&pixscale={pixscale}&size={n}")
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    img = _jpeg(r.content)
    if np.ptp(img) < 5:  # blank: outside footprint
        return None
    bands = "griz" if (layer.startswith("ls-dr1") and "grz" not in layer
                       and coord.dec.deg < 32.375) else "grz"
    return {"title": f"Legacy {layer.replace('ls-', '').upper()} {bands}", "img": img,
            "url": url, "survey": "Legacy"}


def fetch_ps1(coord, size, filters=PS1_FILTERS):
    """Pan-STARRS1 stack colour JPEG via the STScI fitscut service. None outside Dec > -30."""
    import requests
    from astropy.table import Table

    if coord.dec.deg < -30.5:
        return None
    r = requests.get("https://ps1images.stsci.edu/cgi-bin/ps1filenames.py",
                     params=dict(ra=f"{coord.ra.deg:.6f}", dec=f"{coord.dec.deg:.6f}",
                                 filters=filters, type="stack"), timeout=60)
    r.raise_for_status()
    tab = Table.read(r.text, format="ascii")
    if len(tab) == 0:
        return None
    fn = {str(f): str(n) for f, n in zip(tab["filter"], tab["filename"])}
    order = sorted(fn, key="yzirg".find)  # red -> blue
    n = int(np.ceil(size / 0.25))         # PS1 stacks: 0.25"/pix
    params = dict(ra=f"{coord.ra.deg:.6f}", dec=f"{coord.dec.deg:.6f}", size=n,
                  format="jpg", asinh="True", autoscale=99.5)
    if len(order) >= 3:
        rgb = (order[0], order[len(order) // 2], order[-1])
        params.update(red=fn[rgb[0]], green=fn[rgb[1]], blue=fn[rgb[2]])
        bands = "".join(reversed(rgb))
    else:  # single band -> greyscale
        params.update(red=fn[order[0]])
        bands = order[0]
    r = requests.get("https://ps1images.stsci.edu/cgi-bin/fitscut.cgi", params=params,
                     timeout=60)
    r.raise_for_status()
    return {"title": f"PS1 {bands}", "img": _jpeg(r.content), "url": r.url, "survey": "PS1"}


def fetch_ground(coord, size, legacy=True, ps1=True, verbose=True):
    """List of ground-based colour panels (dicts with title/img/url)."""
    out = []
    for flag, fn, label in ((legacy, fetch_legacy, "Legacy"), (ps1, fetch_ps1, "PS1")):
        if not flag:
            continue
        try:
            g = fn(coord, size)
            if g is None:
                _log(f"[{label}] no coverage", verbose)
            else:
                out.append(g)
        except Exception as e:
            _log(f"[{label}] failed: {e}", verbose)
    return out


def save_ground(ground, outdir):
    os.makedirs(outdir, exist_ok=True)
    paths = []
    for g in ground:
        from PIL import Image
        p = os.path.join(outdir, f"{g['survey']}_{_safe(g['title'].split(' ', 1)[1])}.jpg")
        Image.fromarray(g["img"]).save(p, quality=95)
        paths.append(p)
    return paths

### 3f. Top-level fetch, batch runner, FITS writer

In [ ]:
def get_cutouts(ra, dec, size=10.0, telescopes=("hst", "jwst", "euclid"), max_per_band=1,
                euclid_external=False, workers=8, https_fallback=False, verbose=True):
    """Return a list of Cut objects (native pixels, MARGIN x size on a side).

    MAST and IRSA are queried concurrently, and each band's cutout is read in its own thread.
    """
    import time
    t0 = time.time()
    coord = parse_coord(ra, dec)
    big = size * MARGIN
    tels = [t.lower() for t in telescopes]
    mast = [t.upper() for t in tels if t in ("hst", "jwst")]

    def _mast():
        try:
            return fetch_mast(coord, big, mast, max_per_band=max_per_band, workers=workers,
                              https_fallback=https_fallback, verbose=verbose)
        except Exception as e:
            _log(f"[MAST] query failed: {e}", verbose)
            return []

    def _euclid():
        try:
            return fetch_euclid(coord, big, include_external=euclid_external, workers=workers,
                                verbose=verbose)
        except Exception as e:
            _log(f"[Euclid] query failed: {e}", verbose)
            return []

    jobs = ([_mast] if mast else []) + ([_euclid] if "euclid" in tels else [])
    cuts = []
    with ThreadPoolExecutor(max_workers=max(1, len(jobs))) as ex:
        for out in ex.map(lambda f: f(), jobs):
            cuts += out
    order = {"JWST": 0, "HST": 1, "Euclid": 2}
    cuts.sort(key=lambda c: (order.get(c.telescope, 9), np.nan_to_num(c.wave_nm, nan=1e9)))
    _log(f"[done] {len(cuts)} cutouts in {time.time() - t0:.1f}s", verbose)
    return cuts


def run_batch(targets, size=10.0, telescopes=("hst", "jwst", "euclid"), max_per_band=1,
              euclid_external=False, legacy=True, ps1=True, outdir="cutouts", target_workers=3,
              workers=8, skip_done=True, https_fallback=False):
    """Process many targets. `targets` = iterable of (ra, dec, name) or a CSV path (ra,dec[,name]).

    Downloads run `target_workers` targets at a time; FITS/PNG writing and plotting happen in
    the calling thread (matplotlib is not thread-safe). Finished targets are skipped if
    `skip_done`, so an interrupted run can simply be restarted.
    """
    import csv
    import time
    from concurrent.futures import as_completed

    if isinstance(targets, (str, os.PathLike)):
        with open(targets, newline="") as fh:  # csv module keeps names as exact strings
            rows = [{k.strip().lower(): v.strip() for k, v in r.items() if k}
                    for r in csv.DictReader(fh)]
        targets = [(r["ra"], r["dec"], r.get("name")) for r in rows]

    todo = []
    for ra, dec, name in targets:
        c = parse_coord(ra, dec)
        name = name or f"J{c.ra.deg:09.5f}{c.dec.deg:+09.5f}"
        png = os.path.join(outdir, _safe(name), f"{_safe(name)}_cutouts.png")
        if skip_done and os.path.exists(png):
            print(f"skip {name} (done)")
            continue
        todo.append((ra, dec, name, c, png))

    def _fetch(t):
        ra, dec, name, c, png = t
        t0 = time.time()
        cs = get_cutouts(ra, dec, size, telescopes, max_per_band, euclid_external,
                         workers=workers, https_fallback=https_fallback, verbose=False)
        gr = fetch_ground(c, size, legacy=legacy, ps1=ps1, verbose=False)
        return cs, gr, time.time() - t0

    T0, b0 = time.time(), STATS["bytes"]
    with ThreadPoolExecutor(max_workers=target_workers) as ex:
        futs = {ex.submit(_fetch, t): t for t in todo}
        for k, fut in enumerate(as_completed(futs), 1):
            ra, dec, name, c, png = futs[fut]
            try:
                cs, gr, dt = fut.result()
            except Exception as e:
                print(f"[{k}/{len(todo)}] {name}: FAILED ({e})")
                continue
            save_fits(cs, c, size, os.path.dirname(png))
            save_ground(gr, os.path.dirname(png))
            plot_cutouts(cs, ra, dec, size, ground=gr, outfile=png, title=name)
            print(f"[{k}/{len(todo)}] {name}: {len(cs)} space cutouts, "
                  f"{'+'.join(g['survey'] for g in gr) or 'no ground'}, fetch {dt:.1f}s", flush=True)
    print(f"batch: {len(todo)} targets in {time.time() - T0:.0f}s, "
          f"~{(STATS['bytes'] - b0) / 2**20:.0f} MB transferred")


def save_fits(cuts, coord, size, outdir):
    os.makedirs(outdir, exist_ok=True)
    seen, paths = {}, []
    for c in cuts:
        small = Cutout2D(c.data, coord, size * u.arcsec, wcs=c.wcs, mode="partial",
                         fill_value=np.nan)
        hdr = small.wcs.to_header()
        hdr["TELESCOP"] = c.telescope
        hdr["INSTRUME"] = c.instrument
        hdr["FILTER"] = c.band
        hdr["OBS_ID"] = c.obs_id
        if np.isfinite(c.exptime):
            hdr["EXPTIME"] = c.exptime
        hdr["SRCFILE"] = c.source
        hdr["PIXSCALE"] = (c.pixscale, "arcsec/pix")
        base = f"{c.telescope}_{_safe(c.instrument)}_{_safe(c.band)}"
        seen[base] = seen.get(base, 0) + 1
        if seen[base] > 1:
            base += f"_{seen[base]}"
        path = os.path.join(outdir, base + ".fits")
        fits.PrimaryHDU(small.data.astype(np.float32), header=hdr).writeto(path, overwrite=True)
        paths.append(path)
    return paths

### 3g. Display: north-up resampling, Lupton RGB, panel grid with N/E compass

In [ ]:
def _northup_wcs(coord, size, scale):
    n = int(np.ceil(size / scale))
    w = WCS(naxis=2)
    w.wcs.ctype = ["RA---TAN", "DEC--TAN"]
    w.wcs.crval = [coord.ra.deg, coord.dec.deg]
    w.wcs.crpix = [(n + 1) / 2.0, (n + 1) / 2.0]
    w.wcs.cdelt = [-scale / 3600.0, scale / 3600.0]
    return w, (n, n)


def northup(cut, coord, size, scale=None):
    """Resample a cutout onto a north-up, east-left TAN grid centred on `coord`."""
    from reproject import reproject_interp
    w, shape = _northup_wcs(coord, size, scale or cut.pixscale)
    arr, _ = reproject_interp((cut.data, cut.wcs), w, shape_out=shape)
    return arr


def _bgsub(a):
    _, med, std = sigma_clipped_stats(a[np.isfinite(a)], sigma=3, maxiters=5)
    return a - med, std


def make_rgb(cuts, coord, size, Q=10.0, stretch=0.5, pct=99.5):
    """Lupton RGB from the bluest / middle / reddest band of a list of cutouts."""
    from astropy.visualization import make_lupton_rgb
    cs = sorted(cuts, key=lambda c: c.wave_nm)
    b, g, r = (cs[0], cs[len(cs) // 2], cs[-1]) if len(cs) >= 3 else (cs[0], None, cs[-1])
    scale = min(c.pixscale for c in (b, g, r) if c is not None)
    chans = []
    for c in (r, g, b):
        if c is None:
            chans.append(None)
            continue
        a, _ = _bgsub(northup(c, coord, size, scale))
        norm = np.nanpercentile(a, pct)
        chans.append(np.nan_to_num(a / norm if norm > 0 else a))
    if chans[1] is None:
        chans[1] = 0.5 * (chans[0] + chans[2])
    with np.errstate(invalid="ignore", divide="ignore"):
        rgb = make_lupton_rgb(*chans, minimum=0, stretch=stretch, Q=Q)
    bands = "/".join(c.band.split(";")[0] for c in (b, g, r) if c is not None)
    return rgb, bands


def _compass(ax, h, color="w"):
    """N (up) / E (left) arrows in the upper-right corner; extent is [+h, -h, -h, +h]."""
    x0, y0, L = -0.62 * h, 0.55 * h, 0.28 * h
    kw = dict(arrowstyle="-|>", color=color, lw=0.9, mutation_scale=6)
    ax.annotate("", xy=(x0, y0 + L), xytext=(x0, y0), arrowprops=kw)
    ax.annotate("", xy=(x0 + L, y0), xytext=(x0, y0), arrowprops=kw)
    ax.text(x0, y0 + L * 1.12, "N", color=color, fontsize=6, ha="center", va="bottom")
    ax.text(x0 + L * 1.12, y0, "E", color=color, fontsize=6, ha="right", va="center")


def plot_cutouts(cuts, ra, dec, size=10.0, ground=None, outfile=None, ncols=6, rgb=True,
                 title=None):
    """Panel grid, every panel north up / east left (space data resampled with its WCS).

    Order: space RGB composites, ground colour images (Legacy, PS1), single space bands.
    """
    import matplotlib.pyplot as plt
    from astropy.visualization import AsinhStretch, ImageNormalize, ManualInterval

    coord = parse_coord(ra, dec)
    panels = []  # (title, image, is_rgb, origin, note)
    if rgb:
        for tel in ("JWST", "HST", "Euclid"):
            tc = [c for c in cuts if c.telescope == tel]
            if len(tc) >= 2:
                try:
                    img, bands = make_rgb(tc, coord, size)
                    panels.append((f"{tel} RGB {bands}", img, True, "lower", ""))
                except Exception as e:
                    print(f"[RGB] {tel} failed: {e}")
    for g in ground or []:  # JPEGs: row 0 is north
        panels.append((f"{g['title']} (ground)", g["img"], True, "upper", ""))
    for c in cuts:
        a, std = _bgsub(northup(c, coord, size))
        vmax = max(np.nanpercentile(a, 99.7), 5 * std)
        norm = ImageNormalize(interval=ManualInterval(-2 * std, vmax), stretch=AsinhStretch(0.05))
        panels.append((c.label, (a, norm), False, "lower", c.obs_id))

    if not panels:
        print("nothing to plot")
        return None
    ncols = min(ncols, len(panels))
    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.8 * nrows), squeeze=False,
                             layout="constrained")
    h = size / 2.0
    ext = [h, -h, -h, h]  # east left
    for ax, (t, img, is_rgb, origin, note) in zip(axes.flat, panels):
        if is_rgb:
            ax.imshow(img, origin=origin, extent=ext, interpolation="nearest")
        else:
            a, norm = img
            ax.imshow(a, origin="lower", extent=ext, cmap="gray", norm=norm,
                      interpolation="nearest")
        for sx, sy in ((1, 0), (-1, 0), (0, 1), (0, -1)):  # crosshair ticks
            ax.plot([0.12 * h * sx, 0.25 * h * sx], [0.12 * h * sy, 0.25 * h * sy],
                    color="lime", lw=0.8)
        ax.plot([-0.9 * h, -0.9 * h + 1.0], [-0.88 * h] * 2, color="w", lw=1.5)  # 1" bar
        ax.text(-0.9 * h + 0.5, -0.82 * h, '1"', color="w", ha="center", fontsize=7)
        _compass(ax, h)
        if note:
            ax.text(0.03, 0.97, note, transform=ax.transAxes, color="w", fontsize=5.5,
                    va="top", ha="left")
        ax.set_title(t, fontsize=7)
        ax.set_xticks([])
        ax.set_yticks([])
    for ax in axes.flat[len(panels):]:
        ax.axis("off")
    fig.suptitle(title or f"RA={coord.ra.deg:.6f} Dec={coord.dec.deg:+.6f}  "
                 f'({size:g}" box, N up E left)', fontsize=9)
    if outfile:
        fig.savefig(outfile, dpi=150)
        plt.close(fig)
    return fig

## 4. Run one target

In [ ]:
coord = parse_coord(RA, DEC)
name = NAME or f"J{coord.ra.deg:09.5f}{coord.dec.deg:+09.5f}"

b0 = STATS["bytes"]
cuts = get_cutouts(RA, DEC, SIZE, TELESCOPES, MAX_PER_BAND, EUCLID_EXTERNAL, workers=WORKERS)
ground = fetch_ground(coord, SIZE, legacy=LEGACY, ps1=PS1)
print(f"~{(STATS['bytes'] - b0) / 2**20:.0f} MB transferred (space data); ground: "
      + (", ".join(g["title"] for g in ground) or "none"))

pd.DataFrame([dict(telescope=c.telescope, instrument=c.instrument, band=c.band,
                   obs_id=c.obs_id, exptime=c.exptime, pixscale=round(c.pixscale, 4),
                   source=c.source) for c in cuts])

In [ ]:
fig = plot_cutouts(cuts, RA, DEC, SIZE, ground=ground, title=name)
plt.show()

### 4b. Quick look at another target

Fetches and plots in one call; updates `cuts`, `ground`, `coord`, `RA`, `DEC`, `name` so sections 5 and 6 then apply to this target.

In [ ]:
def quicklook(ra, dec, target_name=None, size=None, save=False):
    global RA, DEC, SIZE, coord, name, cuts, ground
    RA, DEC = ra, dec
    SIZE = size or SIZE
    coord = parse_coord(RA, DEC)
    name = target_name or f"J{coord.ra.deg:09.5f}{coord.dec.deg:+09.5f}"
    cuts = get_cutouts(RA, DEC, SIZE, TELESCOPES, MAX_PER_BAND, EUCLID_EXTERNAL, workers=WORKERS)
    ground = fetch_ground(coord, SIZE, legacy=LEGACY, ps1=PS1)
    if save:
        od = os.path.join(OUTDIR, _safe(name))
        save_fits(cuts, coord, SIZE, od)
        save_ground(ground, od)
        plot_cutouts(cuts, RA, DEC, SIZE, ground=ground, title=name,
                     outfile=os.path.join(od, f"{_safe(name)}_cutouts.png"))
    plot_cutouts(cuts, RA, DEC, SIZE, ground=ground, title=name)
    plt.show()

# quicklook(53.1622, -27.7914)
# quicklook("03:32:38.9", "-27:47:29", "my_lens", size=8, save=True)

## 5. Interactive viewer

One space band at a time, north up / east left, with adjustable zoom and stretch. `zoom` is relative to `SIZE` (max 1.4, the fetched margin).

In [ ]:
import ipywidgets as W
from astropy.visualization import AsinhStretch, ImageNormalize, ManualInterval

def _view(i, zoom, asinh_a, vmax_pct, vmin_sigma, cmap):
    c = cuts[i]
    s = SIZE * zoom
    a, std = _bgsub(northup(c, coord, s))
    vmax = max(np.nanpercentile(a, vmax_pct), 3 * std)
    norm = ImageNormalize(interval=ManualInterval(-vmin_sigma * std, vmax),
                          stretch=AsinhStretch(asinh_a))
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(a, origin="lower", extent=[s / 2, -s / 2, -s / 2, s / 2], cmap=cmap,
              norm=norm, interpolation="nearest")
    _compass(ax, s / 2, color="lime")
    ax.set_xlabel("ΔRA [arcsec]  (east left)"); ax.set_ylabel("ΔDec [arcsec]  (north up)")
    ax.set_title(f"{c.label}   {c.obs_id}   {c.pixscale:.3f}\"/pix", fontsize=9)
    plt.show()

if cuts:
    W.interact(_view,
               i=W.Dropdown(options=[(c.label, k) for k, c in enumerate(cuts)], description="band"),
               zoom=W.FloatSlider(value=1.0, min=0.2, max=1.4, step=0.05),
               asinh_a=W.FloatLogSlider(value=0.05, base=10, min=-3, max=0, step=0.1),
               vmax_pct=W.FloatSlider(value=99.7, min=90, max=100, step=0.1),
               vmin_sigma=W.FloatSlider(value=2, min=0, max=5, step=0.5),
               cmap=W.Dropdown(options=["gray", "gray_r", "magma", "viridis"]))
else:
    print("no space-telescope cutouts")

## 6. Save

To `OUTDIR/<name>/`: native-pixel FITS cutouts (WCS + provenance in header), the Legacy/PS1 JPEGs, and the panel PNG.

In [ ]:
outdir = os.path.join(OUTDIR, _safe(name))
paths = save_fits(cuts, coord, SIZE, outdir) + save_ground(ground, outdir)
png = os.path.join(outdir, f"{_safe(name)}_cutouts.png")
plot_cutouts(cuts, RA, DEC, SIZE, ground=ground, outfile=png, title=name)
print("\n".join(paths + [png]))

## 7. Batch mode

CSV with columns `ra,dec[,name]` (names kept as exact strings). `TARGET_WORKERS` targets are fetched at once; finished targets (PNG exists) are skipped, so an interrupted run can be restarted (`SKIP_DONE=False` to redo). Figures are written to disk, not shown inline.

In [ ]:
TARGET_CSV     = "targets.csv"
TARGET_WORKERS = 3          # targets in flight; 3-4 is polite to MAST/IRSA/PS1
SKIP_DONE      = True #False

run_batch(TARGET_CSV, size=SIZE, telescopes=TELESCOPES, max_per_band=MAX_PER_BAND,
          euclid_external=EUCLID_EXTERNAL, legacy=LEGACY, ps1=PS1, outdir=OUTDIR,
          target_workers=TARGET_WORKERS, workers=WORKERS, skip_done=SKIP_DONE)